In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

## Load data

In [2]:
df_train = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_train.csv")
df_test = pd.read_csv(r"C:\mydata\G8Vitamin\data\final\24082025\processed_test.csv")

In [3]:
columns_remove = [
    'VitaminD',
    'YearStart',
]

In [4]:
df_train = df_train[df_train['milk_consumption']<=3]
df_test = df_test[df_test['milk_consumption']<=3]

In [5]:
df_train.drop(columns=columns_remove, inplace=True)
df_test.drop(columns=columns_remove, inplace=True)

In [6]:
category_columns = [
    'Gender','Race' ,'SmokeFam','label','milk_consumption'
]

In [7]:
df_train.columns

Index(['Gender', 'Age', 'Race', 'familysize', 'PIR', 'BMI',
       'WaistCircumference', 'FastingGlucose', 'ALT', 'AST',
       'AlkalinePhosphotase', 'Triglycerides', 'UricAcid', 'Creatinine',
       'HDLCholesterol', 'LDLCholesterol', 'Hemoglobin', 'Hematocrit',
       'MeanCellVolumn', 'MeanCellHemoglobin', 'RedCellDistributionWidth',
       'PlateletCount', 'MeanPlateletVolume', 'Hba1c', 'SmokeFam',
       'milk_consumption', 'label'],
      dtype='object')

In [ ]:
# unuseful_features = ['familysize','SmokeFam','WaistCircumference','AST','ALT','AlkalinePhosphotase','UricAcid','LDLCholesterol','Hematocrit','MeanCellHemoglobin','PlateletCount', 'MeanPlateletVolume','familysize']

In [8]:
unuseful_features = ['SmokeFam','WaistCircumference','AST','AlkalinePhosphotase','UricAcid','LDLCholesterol','Hematocrit','MeanCellHemoglobin','PlateletCount', 'MeanPlateletVolume','familysize']

In [9]:
df_train.drop(columns=unuseful_features,inplace=True)
df_test = df_test[df_train.columns]

In [10]:
df_train.columns

Index(['Gender', 'Age', 'Race', 'PIR', 'BMI', 'FastingGlucose', 'ALT',
       'Triglycerides', 'Creatinine', 'HDLCholesterol', 'Hemoglobin',
       'MeanCellVolumn', 'RedCellDistributionWidth', 'Hba1c',
       'milk_consumption', 'label'],
      dtype='object')

In [11]:
df_train['label'].value_counts()

label
0.0    12380
1.0     3829
Name: count, dtype: int64

In [12]:
df_test['label'].value_counts()

label
0.0    3193
1.0    1437
Name: count, dtype: int64

## Training:  SMOTE with SVMSMOTE

In [28]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE


X_train_raw = df_train.drop(columns='label')
y_train=df_train['label']
X_test_raw=df_test.drop(columns=['label'])
y_test=df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col !='label']

# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)
classifiers = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.7, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}
# ====== 2) Classifiers ======
#lightGBM: {'classifier__learning_rate': 0.05, 'classifier__max_depth': 6, 'classifier__n_estimators': 120, 'classifier__num_leaves': 31}
classifiers = {
    'LightGBM': LGBMClassifier(n_estimators=120,max_depth= 6,num_leaves=31, learning_rate=0.05, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=80,learning_rate=0.066, random_state=42, verbosity=0),
    'GradiantBoosting':GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42),
    #'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42),
    #'KNN': KNeighborsClassifier(n_neighbors=5),
    'Naive Bayes':GaussianNB(var_smoothing= 1e-10),
    'SVM': SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)
}

# ====== 3) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='auto',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 4) Wrapper class for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

# # ====== 5) Train and evaluate models ======
# results = []

# for name, clf in classifiers.items():
#     print(f"\n🚀 Training {name} with SVMSMOTE...")
    
#     pipeline = ImbPipeline(steps=[
#         ('preprocessor', preprocessor),
#         ('smote', svmsmote),
#         ('classifier', clf)
#     ])
    
#     wrapped_model = ImblearnWrapper(pipeline)
    
#     try:
#         wrapped_model.fit(X_train_raw, y_train)
#         y_pred = wrapped_model.predict(X_test_raw)
#         y_proba = wrapped_model.predict_proba(X_test_raw)
        
#         if len(np.unique(y_test)) == 2:
#             auc = roc_auc_score(y_test, y_proba[:, 1])
#         else:
#             auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
        
#         precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
#         recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
#         f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
#         accuracy = accuracy_score(y_test, y_pred)
#         p_class, r_class, f1_class, support = precision_recall_fscore_support(
#             y_test, y_pred, average=None, zero_division=0
#         )

#         print("Per-class metrics:")
#         for i in range(len(p_class)):
#             print(f"Class {i}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")
#         results.append({
#             'Model': name,
#             'Precision (Macro)': precision,
#             'Recall (Macro)': recall,
#             'F1 Score (Macro)': f1,
#             'Accuracy': accuracy,
#             'AUC': auc
#         })
        
#         print(f"✅ {name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
        
#     except Exception as e:
#         print(f"❌ Error training {name}: {e}")

# # ====== 6) Save results ======
# results_df = pd.DataFrame(results)
# print("\n📊 FINAL RESULTS SUMMARY:")
# print("="*60)
# print(results_df.to_string(index=False, float_format='%.4f'))

# results_df.to_csv("svmsmote_classifiers_results.csv", index=False)
# print("\n✅ Results exported to: svmsmote_classifiers_results.csv")

# # ====== 7) Find best model ======
# best_idx = results_df['F1 Score (Macro)'].idxmax()
# best_model = results_df.loc[best_idx, 'Model']
# best_f1 = results_df.loc[best_idx, 'F1 Score (Macro)']
# print(f"\n🏆 BEST MODEL: {best_model} (F1 Score: {best_f1:.4f})")
# ====== 5) Train and evaluate models ======
results = []           # global metrics
per_class_results = [] # per-class metrics

for name, clf in classifiers.items():
    print(f"\n🚀 Training {name} with SVMSMOTE...")
    
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', svmsmote),
        ('classifier', clf)
    ])
    
    wrapped_model = ImblearnWrapper(pipeline)
    
    try:
        wrapped_model.fit(X_train_raw, y_train)
        y_pred = wrapped_model.predict(X_test_raw)
        y_proba = wrapped_model.predict_proba(X_test_raw)
        
        # ---- Global metrics ----
        if len(np.unique(y_test)) == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
        
        precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
        recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        accuracy = accuracy_score(y_test, y_pred)
        
        # ---- Per-class metrics ----
        p_class, r_class, f1_class, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )
        classes = np.unique(y_test)

        for i, cls in enumerate(classes):
            if len(np.unique(y_test)) == 2:
                y_true_bin = (y_test == cls).astype(int)
                auc_class = roc_auc_score(y_true_bin, y_proba[:, i])
            else:
                auc_class = None

            per_class_results.append({
                "Model": name,
                "Class": cls,
                "Precision": p_class[i],
                "Recall": r_class[i],
                "F1": f1_class[i],
                "Support": support[i],
                "AUC": auc_class
            })

        print("Per-class metrics:")
        for i in range(len(p_class)):
            print(f"Class {classes[i]}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")

        results.append({
            'Model': name,
            'Precision (Macro)': precision,
            'Recall (Macro)': recall,
            'F1 Score (Macro)': f1,
            'Accuracy': accuracy,
            'AUC': auc
        })
        
        print(f"✅ {name} - Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")
        
    except Exception as e:
        print(f"❌ Error training {name}: {e}")


# ====== 6) Save results ======
results_df = pd.DataFrame(results)
per_class_df = pd.DataFrame(per_class_results)

print("\n📊 FINAL RESULTS SUMMARY:")
print("="*60)
print(results_df.to_string(index=False, float_format='%.4f'))

results_df.to_csv("svmsmote_classifiers_results.csv", index=False)
per_class_df.to_csv("svmsmote_classifiers_perclass.csv", index=False)

print("\n✅ Results exported to: svmsmote_classifiers_results.csv")
print("✅ Per-class results exported to: svmsmote_classifiers_perclass.csv")


# ====== 7) Find best model ======
best_idx = results_df['F1 Score (Macro)'].idxmax()
best_model = results_df.loc[best_idx, 'Model']
best_f1 = results_df.loc[best_idx, 'F1 Score (Macro)']
print(f"\n🏆 BEST MODEL: {best_model} (F1 Score: {best_f1:.4f})")



🚀 Training LightGBM with SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001811 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Per-class metrics:
Class 0.0: P=0.8221, R=0.6978, F1=0.7549, Support=3193
Class 1.0: P=0.4974, R=0.6646, F1=0.5690, Support=1437
✅ XGBoost - Accuracy: 0.6875, F1: 0.6619, AUC: 0.7401

🚀 Training GradiantBoosting with SVMSMOTE...
Per-class metrics:
Class 0.0: P=0.8253, R=0.6702, F1=0.7397, Support=3193
Class 1.0: P=0.4831, R=0.6848, F1=0.5665, Support=1437
✅ GradiantBoosting - Accuracy: 0.6747, F1: 0.6531, AUC: 0.7387

🚀 Training RandomForest with SVMSMOTE...
Per-class metrics:
Class 0.0: P=0.7810, R=0.8118, F1=0.7961, Support=3193
Class 1.0: P=0.5416, R=0.4941, F1=0.5167, Support=1437
✅ RandomForest - Accuracy: 0.7132, F1: 0.6564, AUC: 0.7321

🚀 Training GradientBoosting with SVMSMOTE...
Per-class metrics:
Class 0.0: P=0.8275, R=0.6759, F1=0.7440, Support=3193
Class 1.0: P=0.4881, R=0.6868, F1=0.5707, Support=1437
✅ GradientBoosting - Accuracy: 0.6793, F1: 0.6573, AUC: 0.7409

🚀 Training Naive Bayes with SVMSMOTE...
Per-class metrics:
Class 0.0: P=0.8109, R=0.6070, F1=0.6943, Support=3

In [13]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
import time
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, 
    roc_auc_score, confusion_matrix
)
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE


# ====== 0) Train/test split ======
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race','milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col !='label']

# ====== 1) Preprocessor ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)

# ====== 2) Classifiers ======
classifiers = {
    'LightGBM': LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0),
    'GradiantBoosting': GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, random_state=42),
    'Naive Bayes': GaussianNB(var_smoothing=1e-10),
    'SVM': SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)
}

# ====== 3) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='auto',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 4) Wrapper for imblearn pipelines ======
class ImblearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def fit(self, X, y):
        self.pipeline.fit(X, y)
        return self
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)


# ====== 5) Train & evaluate ======
rows = []

for name, clf in classifiers.items():
    print(f"\n🚀 Training {name} with SVMSMOTE...")
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', svmsmote),
        ('classifier', clf)
    ])
    wrapped_model = ImblearnWrapper(pipeline)

    try:
        start = time.time()
        wrapped_model.fit(X_train_raw, y_train)
        train_time = time.time() - start

        y_pred = wrapped_model.predict(X_test_raw)
        y_proba = wrapped_model.predict_proba(X_test_raw)

        # ===== Global metrics =====
        acc = accuracy_score(y_test, y_pred)
        f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        # AUC
        if len(np.unique(y_test)) == 2:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        else:
            auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

        # ===== Per-class metrics =====
        cm = confusion_matrix(y_test, y_pred, labels=[0,1])
        classes = [0,1]
        for i, cls in enumerate(classes):
            TP = cm[i,i]
            FN = cm[i,:].sum() - TP
            FP = cm[:,i].sum() - TP
            TN = cm.sum() - (TP+FN+FP)

            PPV = TP/(TP+FP) if (TP+FP)>0 else 0  # Precision
            NPV = TN/(TN+FN) if (TN+FN)>0 else 0
            SEN = TP/(TP+FN) if (TP+FN)>0 else 0  # Recall
            SPE = TN/(TN+FP) if (TN+FP)>0 else 0

            # Add row with per-class metrics
            rows.append({
                "Model": name,
                "Label": cls,
                "Training time": round(train_time,4) if cls==0 else None,
                "ACC": round(acc,4) if cls==0 else None,
                "F1_macro": round(f1_macro,4) if cls==0 else None,
                "F1_weighted": round(f1_weighted,4) if cls==0 else None,
                "AUC": round(auc,4) if cls==0 else None,
                "PPV": round(PPV,4),
                "NPV": round(NPV,4),
                "SEN": round(SEN,4),
                "SPE": round(SPE,4),
            })

        print(f"✅ {name} - ACC={acc:.4f}, F1_macro={f1_macro:.4f}, AUC={auc:.4f}")

    except Exception as e:
        print(f"❌ Error training {name}: {e}")


# ====== 6) Save results ======
results_df = pd.DataFrame(rows)

print("\n📊 FINAL TABLE (Excel format-ready):")
print(results_df)

results_df.to_csv("expected_output_alt.csv", index=False)
print("\n✅ Exported to expected_output_table.csv")



🚀 Training LightGBM with SVMSMOTE...
[LightGBM] [Info] Number of positive: 12380, number of negative: 12380
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002216 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4847
[LightGBM] [Info] Number of data points in the train set: 24760, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


✅ XGBoost - ACC=0.6983, F1_macro=0.6677, AUC=0.7442

🚀 Training GradiantBoosting with SVMSMOTE...
✅ GradiantBoosting - ACC=0.6823, F1_macro=0.6577, AUC=0.7398

🚀 Training RandomForest with SVMSMOTE...
✅ RandomForest - ACC=0.7121, F1_macro=0.6545, AUC=0.7284

🚀 Training GradientBoosting with SVMSMOTE...
✅ GradientBoosting - ACC=0.6812, F1_macro=0.6573, AUC=0.7418

🚀 Training Naive Bayes with SVMSMOTE...
✅ Naive Bayes - ACC=0.6298, F1_macro=0.6143, AUC=0.6928

🚀 Training SVM with SVMSMOTE...
✅ SVM - ACC=0.6635, F1_macro=0.6405, AUC=0.7194

📊 FINAL TABLE (Excel format-ready):
               Model  Label  Training time     ACC  F1_macro  F1_weighted  \
0           LightGBM      0         6.3735  0.6955    0.6633       0.7028   
1           LightGBM      1            NaN     NaN       NaN          NaN   
2            XGBoost      0         4.5082  0.6983    0.6677       0.7059   
3            XGBoost      1            NaN     NaN       NaN          NaN   
4   GradiantBoosting      0        

In [ ]:
# --- IMPORTS ---
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score, precision_recall_fscore_support
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SVMSMOTE
from sklearn.ensemble import GradientBoostingClassifier
# ========== 0) Assume dataset split ==========
X_train_raw = df_train.drop(columns='label')
y_train = df_train['label']
X_test_raw = df_test.drop(columns=['label'])
y_test = df_test['label']

categorical_cols = ['Gender','Race', 'milk_consumption']
numeric_cols = [col for col in df_train.columns if col not in categorical_cols and col != 'label']

# ====== 1) Preprocessor for numerical + categorical columns ======
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    remainder='passthrough'
)
dt_setups = {
    "Shallow": DecisionTreeClassifier(criterion="gini", max_depth=5, min_samples_split=10, min_samples_leaf=4, random_state=42),
    "Balanced": DecisionTreeClassifier(criterion="entropy", max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    "Deep": DecisionTreeClassifier(criterion="gini", max_depth=20, min_samples_split=2, min_samples_leaf=1, random_state=42),
    "Randomized": DecisionTreeClassifier(criterion="log_loss", splitter="random", max_depth=None, min_samples_split=20, min_samples_leaf=6, random_state=42),
    "WideFeatures": DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=30, min_samples_split=5, min_samples_leaf=2, max_features="sqrt", random_state=42)
}



gbc_setups = {
    "Balanced": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
    "ShallowFast": GradientBoostingClassifier(n_estimators=80, learning_rate=0.2, max_depth=2, subsample=0.8, random_state=42),
    "Regularized": GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=4, min_samples_split=20, min_samples_leaf=5, random_state=42),
    "Deep": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.9, random_state=42),
    "RobustSubsample": GradientBoostingClassifier(n_estimators=150, learning_rate=0.08, max_depth=5, subsample=0.7, max_features="sqrt", random_state=42),
    "Conservative": GradientBoostingClassifier(n_estimators=500, learning_rate=0.01, max_depth=3, subsample=0.8, max_features="log2", random_state=42)
}

rf_setups = {
    "Balanced": RandomForestClassifier(
        n_estimators=100, max_depth=None, random_state=42
    ),
    "ShallowFast": RandomForestClassifier(
        n_estimators=50, max_depth=5, max_features="sqrt", random_state=42
    ),
    "Deep": RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        min_samples_leaf=2, random_state=42
    ),
    "Regularized": RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_split=10,
        min_samples_leaf=4, max_features=0.6, random_state=42
    ),
    "Conservative": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=20,
        min_samples_leaf=10, max_features="log2", random_state=42
    ),
    "RobustSubsample": RandomForestClassifier(#no ok
        n_estimators=150, max_depth=15, min_samples_split=5,
        min_samples_leaf=3, bootstrap=True, max_features="sqrt",
        random_state=42
    ),
    "Lightweight": RandomForestClassifier(
        n_estimators=50, max_depth=8, max_features=0.5, random_state=42
    ),
    "Heavy": RandomForestClassifier(
        n_estimators=1000, max_depth=None, min_samples_split=2,
        min_samples_leaf=1, max_features=None, random_state=42, n_jobs=-1
    ),
}

#Best: GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)
base_learners = [
    ('lightgbm', LGBMClassifier(n_estimators=120, max_depth=6, num_leaves=31, learning_rate=0.05, random_state=42)),
    ('xgboost', XGBClassifier(n_estimators=80, learning_rate=0.066, random_state=42, verbosity=0)),
    #('gb', GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, random_state=42)),
    ('RandomForest', rf_setups['Regularized']),
    # ('dt', dt_setups['Shallow']),  
    # ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42)),
    # ('SVM', SVC(kernel='rbf', C=10, gamma=0.01, class_weight='balanced', probability=True, random_state=42)),
    ('nb', GaussianNB(var_smoothing= 1e-10)),
]


# ====== 3) Meta-learner ======
# meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner = LogisticRegression(
    max_iter=3000,
    solver='saga',          # saga supports L1, L2, elasticnet
    penalty='elasticnet',   # mix between L1 and L2
    l1_ratio=0.6,           # 0.0 -> pure L2, 1.0 -> pure L1
    C=1.0,                  # regularization strength
    class_weight='balanced',
    random_state=42
)
# ====== 4) Stacking classifier ======
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    stack_method="predict_proba",  # use probabilities from base models
    n_jobs=-1
)

# ====== 5) SVMSMOTE setup ======
svmsmote = SVMSMOTE(
    random_state=42,
    sampling_strategy='minority',
    k_neighbors=20,
    m_neighbors=40,
    svm_estimator=SVC(kernel="rbf", gamma="scale", C=1.0)
)

# ====== 6) Full pipeline with SVMSMOTE + Stacking ======
stacking_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote1', svmsmote),
    ('classifier', stacking_clf)
])


# ====== 7) Train and evaluate ======
print("\n🚀 Training Ensemble Stacking Model with SVMSMOTE...")
stacking_pipeline.fit(X_train_raw, y_train)

y_pred = stacking_pipeline.predict(X_test_raw)
y_proba = stacking_pipeline.predict_proba(X_test_raw)

# Evaluate
if len(np.unique(y_test)) == 2:
    auc = roc_auc_score(y_test, y_proba[:, 1])
else:
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')

precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_w = f1_score(y_test, y_pred, average='weighted', zero_division=0)
accuracy = accuracy_score(y_test, y_pred)
p_class, r_class, f1_class, support = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)
print(f1_w)
print("\n📊 Per-class metrics:")
for i in range(len(p_class)):
    print(f"Class {i}: P={p_class[i]:.4f}, R={r_class[i]:.4f}, F1={f1_class[i]:.4f}, Support={support[i]}")

print("\n✅ Ensemble Stacking Results")
print(f"Accuracy: {accuracy:.4f}, F1: {f1:.4f}, AUC: {auc:.4f}")



🚀 Training Ensemble Stacking Model with SVMSMOTE...
0.7183369289160212

📊 Per-class metrics:
Class 0: P=0.8022, R=0.7811, F1=0.7915, Support=3193
Class 1: P=0.5404, R=0.5720, F1=0.5558, Support=1437

✅ Ensemble Stacking Results
Accuracy: 0.7162, F1: 0.6736, AUC: 0.7363


c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\mydata\G8Vitamin\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
